In [1]:
# 1. Define the raw download URL for the v0.01 training data
# This is extracted directly from the original script's _DL_URL logic
DOWNLOAD_URL = "https://s3.amazonaws.com/datasets.huggingface.co/SpeechCommands/v0.01/v0.01_train.tar.gz"

print(f"Starting direct download of {DOWNLOAD_URL}...")

# 2. Download the file using wget
!wget -q $DOWNLOAD_URL -O speech_commands_train.tar.gz

print("Download complete. Starting extraction...")

# 3. Extract the contents of the archive into a folder named 'v0.01_train'
!mkdir v0.01_train
!tar -xzf speech_commands_train.tar.gz -C v0.01_train

print("Extraction complete. The audio files are now in the 'v0.01_train' folder.")

Starting direct download of https://s3.amazonaws.com/datasets.huggingface.co/SpeechCommands/v0.01/v0.01_train.tar.gz...
Download complete. Starting extraction...
mkdir: cannot create directory ‘v0.01_train’: File exists
Extraction complete. The audio files are now in the 'v0.01_train' folder.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.


In [3]:
import os
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset



class AudioKwsDataset(Dataset):
    def __init__(
        self,
        root_dir,
        sample_rate=16000,
        n_mels=40,
        exclude_label="_unknown_"
    ):
        self.root_dir = root_dir
        self.sample_rate = sample_rate
        self.samples = []

        # TODO 1:
        # Initialize a MelSpectrogram transform.
        # Hint:
        # - Use a window length of 25 ms
        # - Use a hop length of 10 ms
        # - Number of mel bins = n_mels
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=int(0.025*sample_rate),
            win_length=int(0.025*sample_rate),
            hop_length=int(0.01*sample_rate),
            n_mels=n_mels
        )
        # TODO 2:
        # List all subdirectories inside root_dir
        # Exclude the folder corresponding to `exclude_label`
        all_dirs = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]

        if exclude_label in all_dirs:
            all_dirs.remove(exclude_label)

        # Map label names to integer IDs
        self.label_to_idx = {
            name: idx for idx, name in enumerate(all_dirs)
        }

        # TODO 3:
        # Populate self.samples as (filepath, label_index)
        for label_name in all_dirs:
            dir_path = os.path.join(root_dir, label_name)
            label_idx = self.label_to_idx[label_name]
            for file in os.listdir(dir_path):
                if file.endswith(".wav"):
                    self.samples.append(
                        (os.path.join(dir_path, file), label_idx)
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        # Load audio waveform

        waveform, sr = torchaudio.load(path)

        # TODO 4:
        # Resample the waveform if sampling rate does not match self.sample_rate
        if sr != self.sample_rate:
            waveform = T.Resample(orig_freq=sr, new_freq=self.sample_rate)(waveform)

        # TODO 5:
        # Ensure waveform length is exactly 1 second
        # - Trim if longer
        # - Zero-pad if shorter
        if waveform.shape[1] > self.sample_rate:
            waveform = waveform[ : ,  :self.sample_rate]
        else:
            waveform = nn.functional.pad(
                waveform,
                (0, self.sample_rate - waveform.shape[1])
            )

        # TODO 6:
        # Convert waveform to mel spectrogram
        mel_spec = self.mel_transform(waveform)

        # TODO 7:
        # Apply log scaling for numerical stability
        mel_spec = torch.log(mel_spec + 1e-9)

        return mel_spec, label


In [4]:
import torch
import torch.nn as nn


class Lenet5(nn.Module):
    def __init__(self, num_classes):
        super(Lenet5, self).__init__()

        # Input shape: (batch_size, 1, 40, 101)

        # TODO 1:
        # Define the first convolution layer
        # - Input channels = 1
        # - Output channels = 6
        # - Kernel size = 5
        # - Padding chosen so spatial resolution is preserved
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=6,
            kernel_size=5,
            padding=2
        )

        # TODO 2:
        # Define a 2x2 max pooling layer with stride 2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # TODO 3:
        # Define the second convolution layer
        # - Input channels should match conv1 output
        # - Kernel size = 5
        self.conv2 = nn.Conv2d(
            in_channels=6,
            out_channels=16,
            kernel_size=5
        )

        # TODO 4:
        # Compute the correct flatten size after:
        # Conv1 -> Pool -> Conv2 -> Pool
        # 1, 40, 101 -> 6, 40, 101 -> 6, 20, 50 -> 16, 16, 46 -> 16, 8, 23
        # Hint:
        # Input size = (40, 101)
        # Pooling halves spatial dimensions
        flatten_size = 16*8*23

        self.fc1 = nn.Linear(flatten_size, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

        # TODO 5:
        # Define a ReLU activation
        self.relu = nn.ReLU()

    def forward(self, x):
        # TODO 6:
        # Apply Conv1 -> ReLU -> Pool
        x = self.pool(self.relu(self.conv1(x)))

        # TODO 7:
        # Apply Conv2 -> ReLU -> Pool
        x = self.pool(self.relu(self.conv2(x)))

        # TODO 8:
        # Flatten the tensor for the fully connected layers
        x = x.flatten(1)

        # TODO 9:
        # Apply fully connected layers with ReLU activations
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        # Output layer (no activation)
        x = self.fc3(x)

        return x


In [5]:
root_dir='/content/drive/MyDrive/Colab Notebooks/v0.01_train'
full_dataset = AudioKwsDataset(root_dir)

In [6]:
# Get the dictionary from the original dataset and flip it to index:name
# Your AudioKwsDataset has label_to_idx = {'yes': 0, 'no': 1, ...}
idx_to_label = {v: k for k, v in full_dataset.label_to_idx.items()}

# Create a sorted list based on the indices (0, 1, 2... 18)
class_names = [idx_to_label[i] for i in range(len(idx_to_label))]
print(f"Found {len(class_names)} classes: {class_names[:5]}...")

Found 31 classes: ['wow', '_silence_', 'zero', 'yes', 'six']...


In [7]:
import shutil
import os

# 1. Define paths
drive_path = '/content/drive/MyDrive/Colab Notebooks/v0.01_train' # Update this!
local_path = '/content/v0.01_train_local'

# 2. Copy the entire folder from Drive to Local Colab SSD
if not os.path.exists(local_path):
    print("Moving data to local SSD for high-speed training...")
    shutil.copytree(drive_path, local_path)
    print("Transfer complete!")
else:
    print("Data already exists on local SSD.")

Data already exists on local SSD.


In [8]:
from tqdm.notebook import tqdm
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
import torchaudio



# -------------------------------------------------
# 1. Configuration (Performance-Critical Section)
# -------------------------------------------------

# TODO 1:
# Get the number of available CPU cores
num_cpus = os.cpu_count()
print(f"Using {num_cpus} CPU cores for preprocessing...")

# TODO 2:
# Select GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TODO 3:
# Set dataset path
data_path = '/content/v0.01_train_local'


# -------------------------------------------------
# 2. Dataset Initialization and Splitting
# -------------------------------------------------

# TODO 4:
# Initialize the full keyword spotting dataset
full_dataset = AudioKwsDataset(root_dir=data_path)

# TODO 5:
# Determine number of output classes
num_classes = len(full_dataset.label_to_idx)

# TODO 6:
# Split dataset into 80% training and 20% testing
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])


# -------------------------------------------------
# 3. DataLoaders (Optimize for Speed)
# -------------------------------------------------

train_loader = DataLoader(
    train_data,
    batch_size=32,        # TODO 7: Choose an appropriate batch size
    shuffle=True,
    num_workers=num_cpus,       # TODO 8: Use multiple CPU workers
    pin_memory=False         # TODO 9: Enable/disable pinned memory
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False,
    num_workers=num_cpus,
    pin_memory=False
)


# -------------------------------------------------
# 4. Model, Optimizer, Loss
# -------------------------------------------------

# TODO 10:
# Initialize the CNN model and move it to the selected device
model = Lenet5(num_classes=num_classes).to(device)

# TODO 11:
# Define optimizer
optimizer = Adam(model.parameters(), lr=0.001)

# TODO 12:
# Define loss function for multi-class classification
loss_fn = nn.CrossEntropyLoss()


# -------------------------------------------------
# 5. Training Loop
# -------------------------------------------------

# TODO 13:
# Create directory to save results if it does not exist
if not os.path.exists('results'):
    os.makedirs('results')


for epoch in range(10):  # TODO 14: Choose number of epochs
    model.train()
    total_loss = 0

    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

    for specs, labels in pbar:

        # TODO 15:
        # Move data to device (enable non-blocking transfers if applicable)
        specs = specs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # TODO 16:
        # Zero gradients
        optimizer.zero_grad()

        # TODO 17:
        # Forward pass
        outputs = model(specs)

        # TODO 18:
        # Compute loss
        loss = loss_fn(outputs, labels)

        # TODO 19:
        # Backpropagation and optimizer step
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # -------------------------------------------------
    # 6. Evaluation Loop
    # -------------------------------------------------

    model.eval()
    correct = 0

    with torch.no_grad():
        for spec, lbl in test_loader:
            spec = spec.to(device, non_blocking=True)
            lbl = lbl.to(device, non_blocking=True)

            # TODO 20:
            # Get predicted class labels
            pred = torch.argmax(model(spec), dim=1)

            correct += (pred == lbl).sum().item()

    acc = 100 * correct / len(test_data)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.2f}%")

    # TODO 21:
    # Save model checkpoint
    torch.save(
        model.state_dict(),
        f"results/lenet_epoch_{epoch+1}.pth"
    )


Using 2 CPU cores for preprocessing...


Epoch 1:   0%|          | 0/1278 [00:00<?, ?it/s]

Epoch 1 | Loss: 1.3192 | Acc: 81.96%


Epoch 2:   0%|          | 0/1278 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.4959 | Acc: 86.26%


...

Epoch 8 | Loss: 0.1283 | Acc: 88.62%


Epoch 9:   0%|          | 0/1278 [00:00<?, ?it/s]

Epoch 9 | Loss: 0.1142 | Acc: 87.81%


Epoch 10:   0%|          | 0/1278 [00:00<?, ?it/s]

Epoch 10 | Loss: 0.1011 | Acc: 89.08%


In [9]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import os

# Make sure Lenet5 and AudioKwsDataset are available
# from models import Lenet5
# from dataprocess import AudioKwsDataset


# TODO 1:
# Select GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def test_model(model_path, data_path):

    # -------------------------------------------------
    # 1. Dataset Initialization
    # -------------------------------------------------

    # TODO 2:
    # Initialize dataset exactly as done during training
    full_dataset = AudioKwsDataset(root_dir=data_path)

    # TODO 3:
    # Determine number of output classes
    num_classes = len(full_dataset.label_to_idx)

    # -------------------------------------------------
    # 2. Reproduce Train/Test Split
    # -------------------------------------------------

    # TODO 4:
    # Compute train and test sizes (80/20 split)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size

    # TODO 5:
    # Use random_split with a fixed seed to ensure reproducibility
    _, test_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    # TODO 6:
    # Create DataLoader for test dataset
    test_dataloader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False
    )

    # -------------------------------------------------
    # 3. Load Model and Weights
    # -------------------------------------------------

    # TODO 7:
    # Initialize model architecture
    model = Lenet5(num_classes)

    # TODO 8:
    # Load saved model weights
    model.load_state_dict(
        torch.load(model_path, map_location=device)
    )

    # TODO 9:
    # Move model to device and set evaluation mode
    model = model.to(device)
    model.eval()

    print(f"Testing model on {len(test_dataset)} samples...")

    pred_tags = []
    true_tags = []

    # -------------------------------------------------
    # 4. Inference Loop
    # -------------------------------------------------

    with torch.no_grad():
        for batch in test_dataloader:

            # TODO 10:
            # Extract input data and labels from batch
            batch_data = batch[0].to(device)
            batch_label = batch[1].to(device)

            # TODO 11:
            # Forward pass through the model
            logits = model(batch_data)

            # TODO 12:
            # Convert logits to predicted class indices
            pred = torch.argmax(logits, dim=1).cpu().tolist()

            # Store predictions and true labels
            pred_tags.extend(pred)
            true_tags.extend(
                batch_label.cpu().tolist()
            )

    # -------------------------------------------------
    # 5. Accuracy Calculation
    # -------------------------------------------------

    # TODO 13:
    # Compute classification accuracy
    correct_num = sum(p == t for p, t in zip(pred_tags, true_tags))
    accuracy = correct_num / len(true_tags)

    print(f"Final Test Accuracy: {accuracy * 100:.2f}%")
    return accuracy


if __name__ == "__main__":

    # TODO 14:
    # Set correct paths to trained model and dataset
    BEST_MODEL_PATH = "results/lenet_epoch_10.pth"
    DATA_DIR = '/content/v0.01_train_local'

    if os.path.exists(BEST_MODEL_PATH):
        test_model(BEST_MODEL_PATH, DATA_DIR)
    else:
        print("Error: Model file not found. Did training finish?")


Testing model on 10219 samples...
Final Test Accuracy: 95.99%


In [10]:
# 3. Initialize Model
model = Lenet5(num_classes=num_classes).to(device)
optimizer = Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [11]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
import os

def verify_model_prediction(sample_idx, model_path, dataset_obj):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 1. Get the Ground Truth (Expected Result)
    audio_path, true_label_id = dataset_obj.samples[sample_idx]
    idx_to_label = {v: k for k, v in dataset_obj.label_to_idx.items()}
    expected_word = idx_to_label[true_label_id]

    # 2. Load the Model and Prediction Logic
    num_classes = len(dataset_obj.label_to_idx)
    model = Lenet5(num_classes=num_classes).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 3. Preprocess Audio
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
    if waveform.shape[1] > 16000:
        waveform = waveform[:, :16000]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, 16000 - waveform.shape[1]))

    mel_transform = T.MelSpectrogram(
        sample_rate=16000, n_fft=400, win_length=400, hop_length=160, n_mels=40
    ).to(device)

    spec = mel_transform(waveform.to(device))
    spec = torch.log(spec + 1e-9).unsqueeze(0)

    # 4. Predict
    with torch.no_grad():
        logits = model(spec)
        probs = F.softmax(logits, dim=1)
        conf, pred_idx = torch.max(probs, dim=1)
        predicted_word = idx_to_label[pred_idx.item()]

    # 5. Show Results
    print("="*30)
    print(f"SAMPLE ID: {sample_idx}")
    print(f"FILE: {os.path.basename(audio_path)}")
    print("-" * 30)
    print(f"EXPECTED (True):  {expected_word}")
    print(f"PREDICTED:       {predicted_word}")
    print(f"CONFIDENCE:      {conf.item()*100:.2f}%")
    print("-" * 30)

    if expected_word == predicted_word:
        print("✅ SUCCESS: Prediction matches!")
    else:
        print("❌ FAILURE: Prediction is incorrect.")
    print("="*30)

# --- EXECUTION ---
# You can change 'sample_idx' to any number to test different files
verify_model_prediction(sample_idx=19150, model_path="results/lenet_epoch_10.pth", dataset_obj=full_dataset)

SAMPLE ID: 19150
FILE: 9acd0254_nohash_0.wav
------------------------------
EXPECTED (True):  five
PREDICTED:       five
CONFIDENCE:      100.00%
------------------------------
✅ SUCCESS: Prediction matches!


In [12]:
import torch
import numpy as np
import os

# -------------------------------------------------
# 1. Configuration
# -------------------------------------------------

# TODO 1:
# Path to the trained (distilled) model checkpoint
MODEL_PATH = "results/lenet_epoch_10.pth"

# TODO 2:
# Path to output text file where parameters will be saved
OUTPUT_FILE = "lenet5_parameters.txt"


# -------------------------------------------------
# 2. Determine Number of Classes
# -------------------------------------------------

# TODO 3:
# Dataset root directory (same used during training)
data_path = "/content/v0.01_train_local"

# List all class folders
all_dirs = sorted([
    d for d in os.listdir(data_path)
    if os.path.isdir(os.path.join(data_path, d))
])

# TODO 4:
# Remove the "_unknown_" class if present
if "_unknown_" in all_dirs:
    all_dirs.remove("_unknown_")

# TODO 5:
# Compute number of classes
num_classes = len(all_dirs)


# -------------------------------------------------
# 3. Initialize Model and Load Weights
# -------------------------------------------------

# TODO 6:
# Initialize Lenet5 with correct number of classes
model = Lenet5(num_classes)

# TODO 7:
# Load model weights (CPU is sufficient here)
model.load_state_dict(
    torch.load(MODEL_PATH, map_location="cpu")
)

# TODO 8:
# Set model to evaluation mode
model.eval()

# -------------------------------------------------
# 4. NumPy Print Configuration
# -------------------------------------------------

# Ensure full arrays are printed without truncation
np.set_printoptions(
    threshold=np.inf,
    linewidth=200
)

print(f"Exporting floating-point parameters for {num_classes} classes...")


# -------------------------------------------------
# 5. Export Parameters
# -------------------------------------------------

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for name, params in model.named_parameters():

        print(f"Layer: {name} | Shape: {params.shape}")

        # TODO 9:
        # Write layer name and shape
        f.write(f"{name} | shape: {tuple(params.shape)}")

        # TODO 10:
        # Convert parameters to NumPy and write them
        f.write(str(params.detach().cpu().numpy()))

        # Separator for readability
        f.write("\n" + "=" * 80 + "\n")

print(f"Successfully saved parameters to {OUTPUT_FILE}")


Exporting floating-point parameters for 31 classes...
Layer: conv1.weight | Shape: torch.Size([6, 1, 5, 5])
Layer: conv1.bias | Shape: torch.Size([6])
Layer: conv2.weight | Shape: torch.Size([16, 6, 5, 5])
Layer: conv2.bias | Shape: torch.Size([16])
Layer: fc1.weight | Shape: torch.Size([120, 2944])
Layer: fc1.bias | Shape: torch.Size([120])
Layer: fc2.weight | Shape: torch.Size([84, 120])
Layer: fc2.bias | Shape: torch.Size([84])
Layer: fc3.weight | Shape: torch.Size([31, 84])
Layer: fc3.bias | Shape: torch.Size([31])
Successfully saved parameters to lenet5_parameters.txt


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
import os


# -------------------------------------------------
# 1. DISTILLATION PIPELINE
# -------------------------------------------------

def run_distillation(EPOCHS=50, lr=0.001, batch_size=64, alpha=0.3, T=7):

    # TODO 1:
    # Select GPU if available, otherwise CPU
    device = device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    # -------------------------------------------------
    # Dataset Initialization
    # -------------------------------------------------

    # TODO 2:
    # Initialize keyword spotting dataset
    full_dataset = AudioKwsDataset(root_dir=data_path)

    # TODO 3:
    # Determine number of output classes
    num_classes = len(full_dataset.label_to_idx)

    # TODO 4:
    # Split dataset into training and testing sets (80/20)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_data, test_data = random_split(full_dataset, [train_size, test_size])

    # TODO 5:
    # Create DataLoaders
    train_loader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_cpus,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_cpus
    )

    # -------------------------------------------------
    # 2. TEACHER & STUDENT MODELS
    # -------------------------------------------------

        # TODO 6
    from torchvision.models import resnet18
    teacher_model = resnet18(pretrained=True)

    # TODO 7
    teacher_model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )

    # TODO 8
    teacher_model.fc = nn.Linear(
        teacher_model.fc.in_features, num_classes
    )

    teacher_model = teacher_model.to(device)
    teacher_model.eval()

    # TODO 9
    student_model = Lenet5(num_classes=num_classes).to(device)

    # -------------------------------------------------
    # 3. LOSSES & OPTIMIZER
    # -------------------------------------------------

    # TODO 10
    optimizer = torch.optim.Adam(student_model.parameters(), lr=lr)

    # TODO 11
    hard_loss_fn = nn.CrossEntropyLoss()

    # TODO 12
    soft_loss_fn = nn.KLDivLoss(reduction="batchmean")


    # -------------------------------------------------
    # 4. TRAINING LOOP
    # -------------------------------------------------

    max_test_acc = 0.0

    for epoch in range(EPOCHS):
        student_model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

        for specs, labels in pbar:
            specs, labels = specs.to(device), labels.to(device)

            # TODO 13
            with torch.no_grad():
                teacher_logits = teacher_model(specs)

            # TODO 14
            student_logits = student_model(specs)

            # TODO 15
            loss_hard = hard_loss_fn(student_logits, labels)

            # TODO 16
            loss_soft = soft_loss_fn(
                F.log_softmax(student_logits / T, dim=1),
                F.softmax(teacher_logits / T, dim=1)
            ) * (T * T)

            # TODO 17
            loss = alpha * loss_soft + (1 - alpha) * loss_hard


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pbar.set_postfix(loss=f"{loss.item():.4f}")

        # -------------------------------------------------
        # 5. EVALUATION
        # -------------------------------------------------

        student_model.eval()
        correct = 0

        with torch.no_grad():
            for specs, labels in test_loader:
                specs, labels = specs.to(device), labels.to(device)

                # TODO 18:
                # Compute predictions
                preds = student_model(specs).argmax(dim=1)

                correct += (preds == labels).sum().item()

        acc = 100 * correct / len(test_data)

        # TODO 19:
        # Save best-performing distilled model
        if acc > max_test_acc:
            max_test_acc = acc
            torch.save(
                student_model.state_dict(),
                "results/lenet_epoch_10.pth"
            )
            print(f"Best Distilled Model Saved! Acc: {acc:.2f}%")

    print(f"Distillation Finished. Peak Accuracy: {max_test_acc:.2f}%")


# -------------------------------------------------
# 6. ENTRY POINT
# -------------------------------------------------

if __name__ == "__main__":

    # TODO 20:
    # Run distillation with chosen hyperparameters
    run_distillation(
        EPOCHS=50,
        alpha=0.3,
        T=7
    )


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1:   0%|          | 0/639 [00:00<?, ?it/s]

Best Distilled Model Saved! Acc: 78.77%


Epoch 2:   0%|          | 0/639 [00:00<?, ?it/s]

Best Distilled Model Saved! Acc: 84.39%


Epoch 3:   0%|          | 0/639 [00:00<?, ?it/s]

Best Distilled Model Saved! Acc: 87.78%


Epoch 4:   0%|          | 0/639 [00:00<?, ?it/s]

Best Distilled Model Saved! Acc: 88.32%


Epoch 5:   0%|          | 0/639 [00:00<?, ?it/s]

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split


# -------------------------------------------------
# 1. Dataset Path
# -------------------------------------------------

# TODO 1:
# Set the dataset root directory (use local SSD path if available)
data_path = '/content/v0.01_train_local'


# -------------------------------------------------
# 2. Dataset Initialization
# -------------------------------------------------

# TODO 2:
# Instantiate the full dataset using AudioKwsDataset
full_dataset = AudioKwsDataset(root_dir=data_path)


# -------------------------------------------------
# 3. Reproducible Train/Test Split
# -------------------------------------------------

# TODO 3:
# Set random seed for reproducibility
torch.manual_seed(42)

# TODO 4: 80/20 split
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

# TODO 5
train_data, test_data = random_split(
    full_dataset,
    [train_size, test_size]
)


# -------------------------------------------------
# 4. DataLoader Initialization
# -------------------------------------------------

# TODO 6:
# Choose batch size
batch_size = 64

# TODO 7
train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_cpus,
    pin_memory=True
)

# TODO 8
test_loader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_cpus,
    pin_memory=True
)


# -------------------------------------------------
# 5. Verification Output
# -------------------------------------------------

print("Dataset Successfully Loaded!")
print(f"Total Samples: {len(full_dataset)}")
print(f"Training Samples: {len(train_data)}")
print(f"Testing Samples: {len(test_data)}")

# TODO 9:
# Print number of classes
print(f"Number of Classes: {len(full_dataset.label_to_idx)}")

print("-" * 30)

# TODO 10:
# Print class-to-index mapping
print(f"Class Mapping: {full_dataset.label_to_idx}")


In [ ]:
import torch
import torch.nn as nn
import struct
import os
import numpy as np

# 1. SETUP PATHS AND MODEL
PARAM_DIR = "/content/drive/MyDrive/KWS_pytorch/parameters/"
MODEL_PATH = '/content/drive/MyDrive/KWS_pytorch/distilled_res/distilled_lenet5_best.pt'

if not os.path.exists(PARAM_DIR):
    os.makedirs(PARAM_DIR)

# Initialize your specific LeNet-5 architecture
# Note: num_classes must match your dataset (19 based on previous chat)
quant_model = Lenet5(num_classes=19)
quant_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
quant_model.eval()

def float_to_hex_fp16(f):
    """
    Converts a float to 16-bit Big-Endian Hex.
    '>e' ensures the sign bit is at the far left (bit 15),
    matching your Verilog: sign = floatA[15]
    """
    return struct.pack('>e', f).hex()

# 2. QUANTIZATION & EXPORT LOOP
print("Starting Quantization to 16-bit Hex...")

with torch.no_grad():
    for name, module in quant_model.named_modules():
        # Target only layers with trainable parameters
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            file_path = os.path.join(PARAM_DIR, f"{name}.txt")

            # Extract weights and biases
            w_flat = module.weight.detach().cpu().numpy().flatten()
            b_flat = module.bias.detach().cpu().numpy().flatten() if module.bias is not None else []

            print(f"Exporting {name}: {len(w_flat)} weights, {len(b_flat)} biases...")

            with open(file_path, "w", encoding="utf-8") as f:
                # Write weights first
                for val in w_flat:
                    f.write(float_to_hex_fp16(val) + "\n") # \n makes it $readmemh compatible

                # Append biases immediately after
                for val in b_flat:
                    f.write(float_to_hex_fp16(val) + "\n")

print(f"Export Complete. Files saved in: {PARAM_DIR}")

In [ ]:
from torch.utils.data import DataLoader

# Create a loader with num_workers=0 for stable single-sample inference
simple_test_loader = DataLoader(
    test_data,
    batch_size=1,        # Set to 1 for easier single-audio testing
    shuffle=True,        # Shuffle to get a different sample each time
    num_workers=0,       # 0 ensures no multi-processing hangs
    pin_memory=False
)

print(f"simple_test_loader defined with {len(simple_test_loader)} samples.")

In [ ]:
import torch
import torch.nn as nn
import struct
import os
import numpy as np


# -------------------------------------------------
# 1. DEVICE AND MODEL SETUP
# -------------------------------------------------

# TODO 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# TODO 2
quant_model = Lenet5(num_classes=num_classes)

# TODO 3
MODEL_PATH = "/content/drive/MyDrive/KWS_pytorch/res/kws_model_best.pt"

if os.path.exists(MODEL_PATH):
    quant_model.load_state_dict(
        torch.load(MODEL_PATH, map_location=device)
    )
    print("Distilled weights loaded.")

# TODO 4
quant_model = quant_model.to(device)
quant_model.eval()


# -------------------------------------------------
# 2. QUANTIZATION / ENCODING LOGIC
# -------------------------------------------------

# TODO 5
PARAM_DIR = "./verilog_params"
if not os.path.exists(PARAM_DIR):
    os.makedirs(PARAM_DIR)


def float_to_hex_fp16(f):
    # TODO 6
    return struct.pack('>e', np.float16(f)).hex()


# -------------------------------------------------
# 3. EXPORT WEIGHTS AND BIASES
# -------------------------------------------------

print("Exporting weights and biases to Verilog-ready hex...")

with torch.no_grad():
    for name, module in quant_model.named_modules():

        # TODO 7
        if isinstance(module, (nn.Conv2d, nn.Linear)):

            file_path = os.path.join(PARAM_DIR, f"{name}.txt")

            # TODO 8
            w_flat = module.weight.detach().cpu().numpy().flatten()
            b_flat = module.bias.detach().cpu().numpy().flatten() if module.bias is not None else []

            with open(file_path, "w", encoding="utf-8") as f:

                # TODO 9
                for val in w_flat:
                    f.write(float_to_hex_fp16(val) + "\n")

                # TODO 10
                for val in b_flat:
                    f.write(float_to_hex_fp16(val) + "\n")

            print(f" - {name}: {len(w_flat)} weights + {len(b_flat)} biases saved.")


# -------------------------------------------------
# 4. FINAL ACCURACY VERIFICATION
# -------------------------------------------------

def final_accuracy_check(model, loader):
    model.eval()
    correct = 0
    total = 0

    print("\nVerifying exported model accuracy...")
    with torch.no_grad():
        for specs, labels in loader:

            # TODO 11
            specs = specs.to(device)
            labels = labels.to(device)

            outputs = model(specs)

            # TODO 12
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            if total >= 500:
                break

    print(f"Final Software Accuracy: {100 * correct / total:.2f}%")


# -------------------------------------------------
# 5. RUN VERIFICATION
# -------------------------------------------------

# TODO 13
final_accuracy_check(
    quant_model,
    test_loader
)


In [ ]:
import torch
import struct
import numpy as np

def analyze_quantization(model, layer_name='conv1', num_samples=5):
    model.eval()
    # Get the layer module
    module = dict(model.named_modules())[layer_name]

    # Get true weights (Float32)
    true_weights = module.weight.detach().cpu().numpy().flatten()

    print(f"Quantization Analysis for Layer: {layer_name}")
    print(f"{'True Value (F32)':>18} | {'Hex (FP16)':>10} | {'Quantized Value':>18} | {'Error %':>10}")
    print("-" * 65)

    for i in range(num_samples):
        true_val = true_weights[i]

        # 1. Convert to 16-bit Big-Endian Hex (Quantization)
        hex_val = struct.pack('>e', true_val).hex()

        # 2. Convert back to float to see the "Quantized Value"
        # This is what the FPGA actually "sees"
        quant_val = np.frombuffer(bytes.fromhex(hex_val), dtype='>f2').astype(np.float32)[0]

        # 3. Calculate Precision Loss (Error)
        error = abs(true_val - quant_val)
        error_pct = (error / abs(true_val) * 100) if true_val != 0 else 0

        print(f"{true_val:18.8f} | {hex_val:>10} | {quant_val:18.8f} | {error_pct:8.4f}%")

# Run the comparison
analyze_quantization(quant_model, layer_name='conv2')

In [ ]:
import torch
import random
import struct
import os

def final_quantized_check(model, dataset_obj, loader):
    model.eval()
    model.to(device) # Ensure it's on the right device (GPU/CPU)

    # 1. Get the mapping from Index to Label Name
    idx_to_label = {v: k for k, v in dataset_obj.label_to_idx.items()}

    # 2. Grab a random sample from the test set
    # Using the loader ensures the Log-Mel transform is applied correctly
    specs, labels = next(iter(loader))
    idx = random.randint(0, len(specs) - 1)

    input_tensor = specs[idx].unsqueeze(0).to(device)
    true_idx = labels[idx].item()

    # 3. Predict
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = torch.argmax(output, dim=1).item()

    # 4. Show Results
    print("="*45)
    print(f"{'CATEGORY':<15} | {'VALUE':<20}")
    print("-" * 45)
    print(f"{'Expected Word':<15} | {idx_to_label[true_idx]} (ID: {true_idx})")
    print(f"{'Predicted Word':<15} | {idx_to_label[pred_idx]} (ID: {pred_idx})")
    print("="*45)

    if true_idx == pred_idx:
        print("SUCCESS: The quantized model is 100% accurate for this sample.")
    else:
        print("MISMATCH: Check your bias loading or hex order.")

    # 5. GENERATE VERILOG INPUT STIMULUS
    print("\n--- Verilog Testbench Hex Input (First 5 Pixels) ---")
    # Pull pixels back to CPU for formatting
    pixels = input_tensor.flatten().cpu().numpy()
    for i in range(5):
        # '>e' = 16-bit Big-Endian Hex (matches your floatAdd16 input)
        hex_val = struct.pack('>e', pixels[i]).hex()
        print(f"tb_input[{i}] = 16'h{hex_val}; // Float: {pixels[i]:.6f}")

# Execute the final check
# We use full_dataset for labels and simple_test_loader for the audio sample
final_quantized_check(quant_model, full_dataset, simple_test_loader)